# Trabalho Prático 1 - Versão V9 (Pipeline Corrigido)
## Correção Crítica: Integração do Pré-processamento

**Correção do Erro:** O erro `Input X contains NaN` ocorria porque os modelos estavam a receber os dados em bruto. Nesta versão, cada modelo é envolvido num `Pipeline` que executa automaticamente:
1.  **Imputer:** Preenche valores em falta (NaN).
2.  **Encoder:** Transforma texto em números.
3.  **Scaler:** Normaliza os dados.
4.  **Modelo:** Só então o algoritmo (XGBoost/GBR) recebe os dados limpos.

**Estratégia Mantida:** Imputação por Marca + XGBoost Otimizado + Ensemble.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os, joblib, re, random

from sklearn.model_selection import train_test_split, RandomizedSearchCV, KFold
from sklearn.preprocessing import LabelEncoder, RobustScaler, OneHotEncoder
from sklearn.metrics import mean_squared_error
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor, VotingRegressor, GradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from xgboost import XGBRegressor

warnings.filterwarnings('ignore')

# --- CONFIGURAÇÃO GLOBAL ---
MODE = 'full'  # 'quick' ou 'full'
TRAIN_N_JOBS = -1
RANDOM_STATE = 42

if MODE == 'quick':
    CV_FOLDS = 3
    N_ITER = 2
    XGB_ESTIMATORS = 100
else:
    CV_FOLDS = 5
    N_ITER = 15
    XGB_ESTIMATORS = 3000

os.environ['PYTHONHASHSEED'] = str(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

print(f"Ambiente Pronto. Modo: {MODE} | N_Jobs: {TRAIN_N_JOBS}")

In [ ]:
# Carregar Dados
try:
    train_df = pd.read_csv('../data/train.csv')
    test_df = pd.read_csv('../data/test.csv')
except:
    print("❌ Erro: Verifique os ficheiros csv.")

# --- LIMPEZA E EXTRAÇÃO ---
def clean_engine(row):
    engine = str(row['engine'])
    hp, liters, cylinders = np.nan, np.nan, np.nan
    
    # Extração via Regex
    hp_match = re.search(r'(\d+\.?\d*)HP', engine)
    if hp_match: hp = float(hp_match.group(1))
    
    lit_match = re.search(r'(\d+\.?\d*)L', engine)
    if not lit_match: lit_match = re.search(r'(\d+\.?\d*) Liter', engine)
    if lit_match: liters = float(lit_match.group(1))
    
    cyl_match = re.search(r'V(\d+)', engine)
    if not cyl_match: cyl_match = re.search(r'(\d+) Cylinder', engine)
    if cyl_match: cylinders = float(cyl_match.group(1))
    
    return pd.Series([hp, liters, cylinders])

def map_transmission(trans):
    t = str(trans).lower()
    if 'auto' in t or 'a/t' in t: return 'Automatic'
    if 'manual' in t or 'm/t' in t: return 'Manual'
    if 'cvt' in t: return 'CVT'
    return 'Other'

def clean_df(df):
    data = df.copy()
    # 1. Normalizar Mileage
    if 'mileage' in data.columns:
        if data['mileage'].dtype == 'O':
             data['milage'] = data['mileage'].astype(str).str.replace(',', '').str.extract(r'(\d+)')[0].astype(float)
        else:
             data['milage'] = data['mileage']
        if 'milage' != 'mileage':
            data.drop(columns=['mileage'], inplace=True, errors='ignore')
    elif 'milage' in data.columns:
        if data['milage'].dtype == 'O':
            data['milage'] = data['milage'].astype(str).str.replace(',', '').str.extract(r'(\d+)')[0].astype(float)
        
    # 2. Extração Motor
    data[['HP', 'Liters', 'Cylinders']] = data.apply(clean_engine, axis=1)
    # 3. Idade
    data['age'] = 2025 - pd.to_numeric(data['model_year'], errors='coerce')
    # 4. Transmissão
    data['transmission_grp'] = data['transmission'].apply(map_transmission)
    
    return data

train_df = clean_df(train_df)
test_df = clean_df(test_df)

# --- IMPUTAÇÃO POR MARCA ---
brand_hp_map = train_df.groupby('brand')['HP'].median()
brand_lit_map = train_df.groupby('brand')['Liters'].median()
global_hp = train_df['HP'].median()
global_lit = train_df['Liters'].median()

def smart_fill(row, col, mapping, glob):
    if pd.isna(row[col]):
        return mapping.get(row['brand'], glob)
    return row[col]

for df in [train_df, test_df]:
    df['HP'] = df.apply(lambda x: smart_fill(x, 'HP', brand_hp_map, global_hp), axis=1)
    df['Liters'] = df.apply(lambda x: smart_fill(x, 'Liters', brand_lit_map, global_lit), axis=1)
    for c in ['Cylinders', 'milage']:
        df[c] = df[c].fillna(train_df[c].median())
    
    df['log_milage'] = np.log1p(df['milage'])
    df['hp_per_liter'] = df['HP'] / df['Liters'].replace(0, 1.0)
    
    luxury = ['Bugatti','Lamborghini','Ferrari','McLaren','Rolls-Royce','Bentley','Aston Martin','Porsche','Maserati']
    df['is_super_luxury'] = df['brand'].isin(luxury).astype(int)

# --- LABEL ENCODING ---
cat_cols = ['brand', 'model', 'fuel_type', 'transmission_grp', 'ext_col', 'int_col', 'accident', 'clean_title']
for col in cat_cols:
    le = LabelEncoder()
    train_df[col] = train_df[col].astype(str)
    test_df[col] = test_df[col].astype(str)
    full_data = pd.concat([train_df[col], test_df[col]], axis=0)
    le.fit(full_data)
    train_df[col] = le.transform(train_df[col])
    test_df[col] = le.transform(test_df[col])

print("Limpeza e Engenharia Concluídas.")

In [ ]:
# --- DEFINIÇÃO DO PIPELINE E FEATURES ---
features = ['brand', 'model', 'age', 'log_milage', 'HP', 'Liters', 'Cylinders', 'hp_per_liter',
            'fuel_type', 'transmission_grp', 'ext_col', 'int_col', 'accident', 'clean_title', 'is_super_luxury']

X = train_df[features]
y = np.log1p(train_df['price'])

numeric_feats = ['age', 'log_milage', 'HP', 'Liters', 'Cylinders', 'hp_per_liter']
# Nota: Como já fizemos LabelEncoding, tratamos as categóricas como numéricas ou passamos direto.
# Mas para garantir robustez (e evitar o erro de NaN), passamos tudo pelo Imputer.

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', RobustScaler())
        ]), features)
    ])

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
print("Split de dados concluído.")

### Pipeline de Modelação
Para evitar o erro `Input X contains NaN`, cada modelo é inserido num `Pipeline` que inclui o `preprocessor`. Isto garante que os dados são tratados antes de entrarem no modelo.

In [ ]:
models = []

# Nota: O prefixo 'model__' é necessário porque o modelo está dentro de um Pipeline chamado 'model'

# 1. XGBoost
xgb_pipe = Pipeline([('pre', preprocessor), ('model', XGBRegressor(objective='reg:squarederror', n_jobs=TRAIN_N_JOBS, random_state=RANDOM_STATE))])
xgb_params = {
    'model__n_estimators': [XGB_ESTIMATORS],
    'model__learning_rate': [0.01],
    'model__max_depth': [6, 8, 10],
    'model__subsample': [0.7],
    'model__colsample_bytree': [0.7],
    'model__reg_alpha': [0.1],
    'model__reg_lambda': [1.0]
}
models.append(('XGB', xgb_pipe, xgb_params))

# 2. Random Forest
rf_pipe = Pipeline([('pre', preprocessor), ('model', RandomForestRegressor(n_jobs=TRAIN_N_JOBS, random_state=RANDOM_STATE))])
rf_params = {
    'model__n_estimators': [300],
    'model__max_depth': [15, 20],
    'model__min_samples_leaf': [2, 4]
}
models.append(('RF', rf_pipe, rf_params))

# 3. Gradient Boosting
gb_pipe = Pipeline([('pre', preprocessor), ('model', GradientBoostingRegressor(random_state=RANDOM_STATE))])
gb_params = {
    'model__n_estimators': [500],
    'model__learning_rate': [0.05],
    'model__max_depth': [5],
    'model__subsample': [0.8]
}
models.append(('GBR', gb_pipe, gb_params))

# --- TREINO ---
trained_estimators = []
cv = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

print("A iniciar treino com Pipelines...")

for name, pipeline, params in models:
    print(f"\n>>> Treinando {name}...")
    search = RandomizedSearchCV(pipeline, params, n_iter=N_ITER, cv=cv, 
                                scoring='neg_mean_squared_error', n_jobs=TRAIN_N_JOBS, 
                                random_state=RANDOM_STATE)
    
    search.fit(X_train, y_train)
    best = search.best_estimator_
    
    # Validar
    preds = best.predict(X_val)
    rmse = np.sqrt(mean_squared_error(np.expm1(y_val), np.expm1(preds)))
    print(f"  -> RMSE Validação: {rmse:,.0f}")
    
    # Para o Ensemble, guardamos (nome, pipeline_treinado)
    # O pipeline treinado já inclui o preprocessor e o modelo com melhores parametros
    trained_estimators.append((name, best))
    
    try:
        joblib.dump(best, f'{name}_pipeline_v9.joblib')
    except:
        pass

In [ ]:
# --- ENSEMBLE ---
# Nota: O VotingRegressor não aceita Pipelines diretamente se eles tiverem passos de pré-processamento duplicados.
# Como já treinámos os modelos, vamos fazer um ensemble "manual" das previsões ou um Voting simples 
# se usarmos os estimadores internos.

# A forma mais segura aqui para evitar re-treinar ou erros de pipeline:
# Usar os pipelines já treinados (best_estimators) para prever e fazer a média ponderada.

print("\nCalculando Ensemble Ponderado...")

# Pesos: XGB(0.6), RF(0.3), GBR(0.1)
weights = {'XGB': 0.6, 'RF': 0.3, 'GBR': 0.1}

X_test_final = test_df[features] # O pipeline vai tratar disto internamente

final_predictions = np.zeros(len(X_test_final))
val_predictions = np.zeros(len(X_val))

for name, model in trained_estimators:
    weight = weights.get(name, 0)
    print(f" -> A adicionar {name} com peso {weight}")
    
    # Prever Validação (para score)
    val_pred = model.predict(X_val)
    val_predictions += val_pred * weight
    
    # Prever Teste (para submissão)
    test_pred = model.predict(X_test_final)
    final_predictions += test_pred * weight

rmse_ens = np.sqrt(mean_squared_error(np.expm1(y_val), np.expm1(val_predictions)))
print(f"\n🏆 RMSE ENSEMBLE FINAL: {rmse_ens:,.2f}")

# --- SUBMISSÃO ---
final_price = np.expm1(final_predictions)
submission = pd.DataFrame({'id': test_df['id'], 'price': np.clip(final_price, 0, None)})
submission.to_csv('submission.csv', index=False)
print("✅ submission.csv criado com sucesso!")